#Importing Dataset

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d soumikrakshit/classical-music-midi

Dataset URL: https://www.kaggle.com/datasets/soumikrakshit/classical-music-midi
License(s): unknown
  0% 0.00/2.33M [00:00<?, ?B/s]
100% 2.33M/2.33M [00:00<00:00, 1.12GB/s]


In [ ]:
!unzip classical-music-midi.zip

Archive:  classical-music-midi.zip
  inflating: albeniz/alb_esp1.mid    
  inflating: albeniz/alb_esp2.mid    
  inflating: albeniz/alb_esp3.mid    
  inflating: albeniz/alb_esp4.mid    
  inflating: albeniz/alb_esp5.mid    
  inflating: albeniz/alb_esp6.mid    
  inflating: albeniz/alb_se1.mid     
  inflating: albeniz/alb_se2.mid     
  inflating: albeniz/alb_se3.mid     
  inflating: albeniz/alb_se4.mid     
  inflating: albeniz/alb_se5.mid     
  inflating: albeniz/alb_se6.mid     
  inflating: albeniz/alb_se7.mid     
  inflating: albeniz/alb_se8.mid     
  inflating: bach/bach_846.mid       
  inflating: bach/bach_847.mid       
  inflating: bach/bach_850.mid       
  inflating: balakir/islamei.mid     
  inflating: beeth/appass_1.mid      
  inflating: beeth/appass_2.mid      
  inflating: beeth/appass_3.mid      
  inflating: beeth/beethoven_hammerklavier_1.mid  
  inflating: beeth/beethoven_hammerklavier_2.mid  
  inflating: beeth/beethoven_hammerklavier_3.mid  
  inflating: b

#IMPORTS

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

import music21

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

import matplotlib.pyplot as plt


#Dataset Audit

In [ ]:
midi_files = glob.glob("**/*.mid", recursive=True)
len(midi_files)

292

In [ ]:
sample = music21.converter.parse(midi_files[0])
sample.show('text')


/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 2010 by Bernd Krueger'>; getting generic Instrument
  warnings.warn(


{0.0} <music21.metadata.Metadata object at 0x7a821313ea50>
{0.0} <music21.stream.Part 0x7a821313f650>
    {0.0} <music21.stream.Measure 1 offset=0.0>
        {0.0} <music21.instrument.Piano 'Piano right: Piano right'>
        {0.0} <music21.instrument.Piano 'Piano'>
        {0.0} <music21.clef.TrebleClef>
        {0.0} <music21.tempo.MetronomeMark Quarter=122.29>
        {0.0} <music21.key.Key of D major>
        {0.0} <music21.meter.TimeSignature 3/4>
        {0.0} <music21.note.Rest half>
        {2.0} <music21.tempo.MetronomeMark Quarter=117.87>
        {2.0} <music21.note.Note A>
        {2.75} <music21.note.Note F#>
    {3.0} <music21.stream.Measure 2 offset=3.0>
        {0.0} <music21.tempo.MetronomeMark animato Quarter=121.94>
        {0.0} <music21.note.Note D>
        {0.25} <music21.note.Rest dotted-eighth>
        {1.0} <music21.note.Note D>
        {1.25} <music21.note.Rest dotted-eighth>
        {2.0} <music21.note.Note D>
        {2.5} <music21.chord.Chord C#5 D5>
       

In [ ]:
for element in sample.recurse():
    if isinstance(element, music21.instrument.Instrument):
        print(element)


Piano right: Piano right
Piano
Piano left: Piano left
Piano


In [ ]:
notes = []
for element in sample.flat:
    if isinstance(element, music21.note.Note):
        notes.append('note')
    elif isinstance(element, music21.chord.Chord):
        notes.append('chord')

print("Notes:", notes.count('note'))
print("Chords:", notes.count('chord'))


Notes: 1365
Chords: 170


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: Music21DeprecationWarning: .flat is deprecated.  Call .flatten() instead
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
durations = []
for element in sample.flat:
    if isinstance(element, music21.note.Note):
        durations.append(element.duration.quarterLength)

print(set(durations))


{0.75, 0.25, 1.75, 0.5, 1.0, 2.0, 1.5, 1.25, Fraction(1, 3)}


In [ ]:
all_notes = []
all_durations = []

for file in midi_files[:20]:  # check first 20
    midi = music21.converter.parse(file)
    for element in midi.flatten():
        if isinstance(element, music21.note.Note):
            all_notes.append(element.pitch.midi)
            all_durations.append(element.duration.quarterLength)
        elif isinstance(element, music21.chord.Chord):
            all_notes.append(tuple(n.pitch.midi for n in element.notes))
            all_durations.append(element.duration.quarterLength)

print("Unique durations:", sorted(set(all_durations))[:15])
print("Min pitch:", min([n for n in all_notes if type(n)==int]))
print("Max pitch:", max([n for n in all_notes if type(n)==int]))


/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 2009 by Bernd Krueger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 1998 by Bernd Kr\xfcger'>; getting generic Instrument
  warnings.warn(


Unique durations: [Fraction(1, 12), Fraction(1, 6), 0.25, Fraction(1, 3), 0.5, Fraction(2, 3), 0.75, 1.0, 1.25, Fraction(4, 3), 1.5, 1.75, 2.0, 2.25, Fraction(7, 3)]
Min pitch: 31
Max pitch: 89


In [ ]:
from music21 import converter, note, chord

def extract_events(file):
    midi = converter.parse(file)
    events = []

    for element in midi.flatten():
        if isinstance(element, note.Note):
            pitch = str(element.pitch.midi)
            duration = round(element.duration.quarterLength, 1)
            events.append(pitch + "_" + str(duration))

        elif isinstance(element, chord.Chord):
            duration = round(element.duration.quarterLength, 1)
            for n in element.notes:
                pitch = str(n.pitch.midi)
                events.append(pitch + "_" + str(duration))

    return events




In [ ]:
all_events = []

for file in midi_files:
    events = extract_events(file)
    all_events.extend(events)

print("Total events:", len(all_events))
print("Unique tokens:", len(set(all_events)))
#dont run this command please , it takes 20 mins
#here's the op Total events: 481419
#Unique tokens: 30851

/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Copyright \xa9 2004 by Bernd Kr\xfcger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 2007 by Bernd Krueger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Copyright \xa9 1997 by Bernd Krueger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Co

Total events: 717799
Unique tokens: 1845


In [ ]:
20vocab = sorted(set(all_events))
token_to_int = {token: i for i, token in enumerate(vocab)}
int_to_token = {i: token for token, i in token_to_int.items()}


In [ ]:
encoded = [token_to_int[token] for token in all_events]


In [ ]:
seq_length = 50


In [ ]:
X = []
y = []

for i in range(len(encoded) - seq_length):
    X.append(encoded[i:i+seq_length])
    y.append(encoded[i+seq_length])

X = np.array(X)
y = np.array(y)


In [ ]:
vocab_size = 1845


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(seq_length,)),
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=256),

    tf.keras.layers.LSTM(512, return_sequences=True),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.LSTM(512),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(vocab_size, activation='softmax')
])



In [ ]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 50, 256)        │       472,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ (None, 50, 512)        │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 50, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 512)            │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1845)           │       946,485 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,355,573 (20.43 MB)

 Trainable params: 5,355,573 (20.43 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)


In [ ]:
seq_length = 50
stride = 3

X = []
y = []

for i in range(0, len(encoded) - seq_length, stride):
    X.append(encoded[i:i+seq_length])
    y.append(encoded[i+seq_length])

X = np.array(X)
y = np.array(y)

print("Number of sequences:", len(X))


Number of sequences: 239250


In [ ]:
# Shuffle before splitting
indices = np.arange(len(X))
np.random.shuffle(indices)

X = X[indices]
y = y[indices]

# Now split
split = int(0.9 * len(X))

X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print("Train:", X_train.shape)
print("Val:", X_val.shape)



Train: (215325, 50)
Val: (23925, 50)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=128,
    epochs=20
)


Epoch 1/20
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 101s 58ms/step - loss: 5.4833 - val_loss: 4.8840
Epoch 2/20
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 98s 58ms/step - loss: 4.8566 - val_loss: 4.6484
Epoch 3/20
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 98s 58ms/step - loss: 4.6722 - val_loss: 4.4776
Epoch 4/20
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 98s 58ms/step - loss: 4.5372 - val_loss: 4.3700
Epoch 5/20
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 98s 58ms/step - loss: 4.4127 - val_loss: 4.2954
Epoch 6/20
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 98s 58ms/step - loss: 4.3230 - val_loss: 4.1985
Epoch 7/20
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 98s 58ms/step - loss: 4.2294 - val_loss: 4.1081
Epoch 8/20
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 98s 58ms/step - loss: 4.1552 - val_loss: 4.0591
Epoch 9/20
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 99s 59ms/step - loss: 4.0806 - val_loss: 4.0031
Epoch 10/20
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 99s 59ms/step - loss: 4.0172 - val_loss: 3.9616
Epoch 11/20
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 99s 59ms/step - loss: 3.9664 - val_loss: 3.9414
Epoch 1

In [ ]:
model.save("/content/drive/MyDrive/music_lstm_model.keras")


In [ ]:
seq_length = 100

In [ ]:
import numpy as np

def sample_with_temperature(preds, temperature=1.0, top_k=15):
    preds = np.asarray(preds).astype('float64')

    # Temperature scaling
    preds = np.log(preds + 1e-8) / temperature
    preds = np.exp(preds)

    # Top-k filtering
    top_indices = np.argsort(preds)[-top_k:]
    filtered_preds = np.zeros_like(preds)
    filtered_preds[top_indices] = preds[top_indices]
    filtered_preds = filtered_preds / np.sum(filtered_preds)

    return np.random.choice(len(preds), p=filtered_preds)

In [ ]:
def generate_music(model, seed_sequence, num_generate=200, temperature=0.6):
    generated = []
    current_seq = seed_sequence.copy()

    for _ in range(num_generate):
        input_seq = np.array(current_seq[-seq_length:])
        input_seq = input_seq.reshape(1, -1)

        preds = model.predict(input_seq, verbose=0)[0]

        # ---- Repetition penalty ----
        recent_tokens = set(current_seq[-20:])
        for t in recent_tokens:
            preds[t] *= 0.6

        # ---- Pitch jump penalty (STEP 3 HERE) ----
        last_token = current_seq[-1]
        last_pitch = int(int_to_token[last_token].split("_")[0])

        for i in range(len(preds)):
            pitch = int(int_to_token[i].split("_")[0])
            if abs(pitch - last_pitch) > 12:   # more than 1 octave jump
                preds[i] *= 0.5

        # Normalize AFTER all penalties
        preds = preds / np.sum(preds)

        next_token = sample_with_temperature(preds, temperature)

        generated.append(next_token)
        current_seq.append(next_token)

    return generated




In [ ]:
start_index = np.random.randint(0, len(X_train))
seed = list(X_train[start_index])


NameError: name 'X_train' is not defined

In [ ]:
generated_tokens = generate_music(model, seed, num_generate=300, temperature=0.6)


NameError: name 'model' is not defined

In [ ]:
from music21 import stream, note, tempo
from fractions import Fraction

def tokens_to_midi(tokens, int_to_token, output_file="generated_music.mid"):
    midi_stream = stream.Stream()

    # Slow tempo
    midi_stream.append(tempo.MetronomeMark(number=110))  # try 50–60

    for token in tokens:
        token_str = int_to_token[token]
        pitch, duration = token_str.split("_")
        duration = float(Fraction(duration))

        new_note = note.Note(int(pitch))
        new_note.duration.quarterLength = duration
        midi_stream.append(new_note)

    midi_stream.write('midi', output_file)


In [ ]:
tokens_to_midi(generated_tokens, int_to_token, "generated_music.mid")

NameError: name 'generated_tokens' is not defined